# XGBoost Cross-Sectional Return Predictor

Walk-forward backtest using XGBoost on the full S&P 500 universe.
Results (charts + tables) are saved to .

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '../src')

from evaluation import WalkForwardBacktester
from models import XGBoostRankModel
import feature_engineer
import utils

## 1  Data Loading

In [ ]:
df = pd.read_parquet("../data/raw/yahoo_raw.parquet")
df = df.stack(level="Ticker", future_stack=True).reset_index()
df = df.sort_values(["Ticker", "Date"]).set_index("Date")
df.columns.name = None
print(f"Loaded {df['Ticker'].nunique()} tickers, {len(df):,} rows")
df.head()

## 2  Feature Engineering

In [ ]:
FEATURE_CONFIG = {
    "mom_windows":  [(1, 5), (1, 21), (21, 126), (21, 252), (126, 252), (252, 756)],
    "vol_windows":  [21, 63, 252],
    "liq_windows":  [21, 63],
    "high_windows": [252],
    "max_windows":  [21],
}

FEATURE_COLS = [
    "mom_1_5", "mom_1_21", "mom_21_126", "mom_21_252", "mom_126_252", "mom_252_756",
    "vol_21", "downside_dev_21", "vol_63", "downside_dev_63", "vol_252", "downside_dev_252",
    "dollar_volume_21", "dollar_volume_63",
    "dist_252_high", "max_21",
]

df = feature_engineer.compute_momentum_features(df, FEATURE_CONFIG["mom_windows"])
df = feature_engineer.compute_volatility_features(df, FEATURE_CONFIG["vol_windows"])
df = feature_engineer.compute_liquidity(df, FEATURE_CONFIG["liq_windows"])
df = feature_engineer.compute_extreme_features(
    df, high_windows=FEATURE_CONFIG["high_windows"], max_windows=FEATURE_CONFIG["max_windows"]
)
print("Features computed.")

## 3  Target Construction

Forward 21-day cross-sectional rank (pct rank across all stocks at each date).

In [ ]:
df["fwd_return_21"] = (
    df.groupby("Ticker")["Adj Close"]
    .pct_change(21)
    .shift(-21)
)

df["target"] = df.groupby("Date")["fwd_return_21"].rank(pct=True)

# Cross-sectionally rank features to remove outlier sensitivity
df[FEATURE_COLS] = df.groupby("Date")[FEATURE_COLS].rank(pct=True)

# Trim to dates where the longest-window feature is available
t0 = df.reset_index().groupby("Date")["mom_252_756"].count()
t0 = t0[t0 >= 1].index[0]
df = df[df.index > t0]

print(f"Panel shape after trim: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")

## 4  Walk-Forward Backtest

In [ ]:
MODEL_NAME = "xgboost"
RESULTS_DIR = "../results/xgboost"

model = XGBoostRankModel(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.2,
    min_child_weight=100,
    gamma=0.5,
)

backtester = WalkForwardBacktester(
    model=model,
    initial_train_months=84,   # 7-year expanding window
    test_months=12,
    step_months=1,
    embargo_days=21,
)

results = backtester.run(df, feature_cols=FEATURE_COLS, target_col="target")
print(f"
Completed {len(results)} folds.")

## 5  Fold Summary

In [ ]:
fold_summary = backtester.summary(results)
print(fold_summary.to_string(index=False))

## 6  Diagnostics — IC, ICIR, L/S Spread

In [ ]:
ic_series, ls_spread = utils.diagnose(
    results,
    ic_on_every=21,
    save_dir=RESULTS_DIR,
    model_name=MODEL_NAME,
)

## 7  Feature Importance

Aggregate XGBoost feature importance across all folds (average gain).

In [ ]:
from models import XGBoostRankModel
import xgboost as xgb
import pandas as pd

# Re-run one final model on the full dataset to get stable importances
# (last fold model is already in  after the walk-forward)
all_importances = []
for r in results:
    # We stored the model per fold — use the last model for a quick estimate
    pass

# Use the model from the final fold (already fitted)
importance = model.get_feature_importance()
if importance is not None:
    fig = utils.plot_feature_importance(
        importance,
        model_name=MODEL_NAME,
        top_n=len(FEATURE_COLS),
        save_dir=RESULTS_DIR,
    )
    plt.show()
else:
    print("No feature importances available.")

## 8  Per-Fold IC Summary

In [ ]:
import matplotlib.pyplot as plt

fold_ics = []
for r in results:
    import pandas as pd
    from scipy.stats import spearmanr
    df_fold = pd.DataFrame({"pred": r["preds"], "actual": r["actuals"]}, index=r["index"])
    sampled = df_fold.index.unique().sort_values()[::21]
    df_s = df_fold[df_fold.index.isin(sampled)]
    ic = df_s.groupby("Date").apply(lambda x: x["pred"].corr(x["actual"], method="spearman")).mean()
    fold_ics.append({"fold": r["fold"], "test_start": r["test_start"], "mean_ic": ic})

fold_ic_df = pd.DataFrame(fold_ics)
print(fold_ic_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["steelblue" if v > 0 else "tomato" for v in fold_ic_df["mean_ic"]]
ax.bar(fold_ic_df["fold"], fold_ic_df["mean_ic"], color=colors)
ax.axhline(fold_ic_df["mean_ic"].mean(), color="black", linestyle="--",
           label=f'Mean = {fold_ic_df["mean_ic"].mean():.4f}')
ax.set_xlabel("Fold")
ax.set_ylabel("Mean IC")
ax.set_title("Per-Fold Mean IC — XGBoost")
ax.legend()
plt.tight_layout()

import pathlib
pathlib.Path(f"{RESULTS_DIR}/charts").mkdir(parents=True, exist_ok=True)
fig.savefig(f"{RESULTS_DIR}/charts/xgboost_per_fold_ic.png", dpi=150, bbox_inches="tight")
fold_ic_df.to_csv(f"{RESULTS_DIR}/tables/xgboost_per_fold_ic.csv", index=False)
plt.show()